In [ ]:
# ============================
# Full pipeline: single cell
# ============================
# NOTE: If required, run (once) in Colab:
!pip install -q scikit-learn xgboost shap lime imbalanced-learn openpyxl xlrd

import os, time, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from packaging import version
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.base import clone
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

import shap
from lime.lime_tabular import LimeTabularExplainer

# ---------------------
# Config
# ---------------------
RANDOM_STATE = 42
TEST_SIZE = 0.30
TARGET_N = 5269
ARTIFACTS = "artifacts"
os.makedirs(ARTIFACTS, exist_ok=True)
np.random.seed(RANDOM_STATE)
plt.rcParams['figure.figsize'] = (8,5)

# ---------------------
# FAST / FULL switch
# ---------------------
FAST_MODE = False   # set False for full experiments (slower)
if FAST_MODE:
    print("=== FAST MODE (quick debugging) ===")
else:
    print("=== FULL MODE (full experiments) ===")

# ---------------------
# Helper functions
# ---------------------
def evaluate_all(y_true, y_pred, y_score=None):
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0)
    }
    try:
        if y_score is not None:
            out["roc_auc_ovr"] = roc_auc_score(pd.get_dummies(y_true), y_score, average="macro", multi_class="ovr")
        else:
            out["roc_auc_ovr"] = np.nan
    except Exception:
        out["roc_auc_ovr"] = np.nan
    return out

def binarize_series(s: pd.Series):
    """Numeric -> median split; Categorical -> mode vs rest"""
    try:
        if pd.api.types.is_numeric_dtype(s):
            arr = pd.to_numeric(s, errors="coerce")
            thr = np.nanmedian(arr)
            yb = (arr > thr).astype(int).fillna(0).astype(int)
            return yb.values, {"rule": "numeric_median", "threshold": float(thr)}
        else:
            sm = s.astype(str)
            mode_lbl = sm.mode().iloc[0]
            yb = (sm != mode_lbl).astype(int)
            return yb.values, {"rule": "categorical_mode_vs_rest", "negative_class": mode_lbl}
    except Exception as e:
        sm = s.astype(str)
        mode_lbl = sm.mode().iloc[0]
        yb = (sm != mode_lbl).astype(int)
        return yb.values, {"rule": "fallback_mode_vs_rest", "negative_class": mode_lbl}

# ---------------------
# Load dataset (CSV)
# ---------------------
DATA_FILE = "processed_data_1.csv"   # change if needed
if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(f"File not found: {DATA_FILE} — upload it or set DATA_FILE correctly.")

df = pd.read_csv(DATA_FILE, encoding='latin-1')  # try latin-1 fallback
print("Raw shape:", df.shape)
display(df.head())

# ---------------------
# Drop obvious ID-like columns
# ---------------------
drop_like = ["id","user_id","email","name","timestamp"]
for d in drop_like:
    if d in df.columns:
        df.drop(columns=[d], inplace=True)
print("After dropping ids:", df.shape)

# ---------------------
# Detect / set target columns
# ---------------------
# Try to auto-detect; if wrong, edit the mapping below manually.
TARGETS = {}
for c in df.columns:
    lc = c.lower()
    if ('depress' in lc or 'phq' in lc) and 'depression' not in TARGETS:
        TARGETS['depression'] = c
    if ('satisf' in lc or 'swls' in lc or 'life_satisfaction' in lc or ('life' in lc and 'satisfaction' in lc)) and 'life_satisfaction' not in TARGETS:
        TARGETS['life_satisfaction'] = c
    if ('person' in lc or 'bfi' in lc or 'bigfive' in lc or 'big five' in lc) and 'personality' not in TARGETS:
        TARGETS['personality'] = c

print("Auto-detected TARGETS:", TARGETS)
if len(TARGETS) < 1:
    raise RuntimeError("No targets auto-detected — set TARGETS manually to your target column names.")
# If auto-detection is incorrect, you can overwrite TARGETS here, e.g.:
# TARGETS = {'depression':'depression_level', 'life_satisfaction':'swls', 'personality':'bfi_total'}

target_cols = list(TARGETS.values())
feat_df = df.drop(columns=[c for c in target_cols if c in df.columns], errors="ignore")
Y_raw = df[target_cols].copy()

# ---------------------
# Preprocessing (impute before augmentation)
# ---------------------
num_cols = feat_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in feat_df.columns if c not in num_cols]

feat_filled = feat_df.copy()
if num_cols:
    num_med = feat_df[num_cols].median()
    feat_filled[num_cols] = feat_df[num_cols].fillna(num_med)
if cat_cols:
    for c in cat_cols:
        mode_val = feat_df[c].mode().iloc[0] if feat_df[c].notna().any() else "NA"
        feat_filled[c] = feat_df[c].fillna(mode_val)

# ---------------------
# Augmentation -> exactly TARGET_N rows
# ---------------------
n_current = feat_filled.shape[0]
print("Original rows:", n_current)
if n_current < TARGET_N:
    need = TARGET_N - n_current
    rng = np.random.default_rng(RANDOM_STATE)
    idx = rng.integers(0, n_current, size=need)
    feat_syn = feat_filled.iloc[idx].copy()
    if num_cols:
        stds = feat_filled[num_cols].std().replace(0,1e-12)
        noise = pd.DataFrame(rng.normal(0,1,size=(need,len(num_cols))), columns=num_cols)
        feat_syn[num_cols] = feat_syn[num_cols] + 0.10 * noise.values * stds.values
    Y_syn = Y_raw.iloc[idx].reset_index(drop=True)
    feat_aug = pd.concat([feat_filled, feat_syn], ignore_index=True)
    Y_aug = pd.concat([Y_raw, Y_syn], ignore_index=True)
else:
    feat_aug = feat_filled.copy()
    Y_aug = Y_raw.copy()

print("After augmentation rows:", feat_aug.shape[0])

# ---------------------
# Vectorize features (Version-safe OneHotEncoder)
# ---------------------
# OneHotEncoder sparse parameter name changed between sklearn versions
if version.parse(sklearn.__version__) >= version.parse("1.2"):
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
else:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_pipe = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())])
categorical_pipe = Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", ohe)])

ct = ColumnTransformer(transformers=[("num", numeric_pipe, num_cols),
                                     ("cat", categorical_pipe, cat_cols)],
                       remainder="drop")

print("Fitting transformer... (this may take a few seconds)")
X_all = ct.fit_transform(feat_aug)                      # numpy array
feature_names = ct.get_feature_names_out().tolist()
print("Transformed shape:", X_all.shape, "| n_features:", len(feature_names))

# ---------------------
# Prepare model configs (FAST vs FULL)
# ---------------------
if FAST_MODE:
    BASE_MODELS = {
        "LogisticRegression": LogisticRegression(max_iter=500, random_state=RANDOM_STATE),
        "XGBoost": XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                                 n_estimators=20, max_depth=3, learning_rate=0.1, random_state=RANDOM_STATE, n_jobs=2)
    }
    STACK_CV = 2
    RUN_HYBRID = False
else:
    BASE_MODELS = {
        "LogisticRegression": LogisticRegression(max_iter=800, random_state=RANDOM_STATE),
        "DecisionTree": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "SVM": SVC(probability=True, random_state=RANDOM_STATE),
        "GaussianNB": GaussianNB(),
        "XGBoost": XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                                 n_estimators=100, max_depth=4, learning_rate=0.1, random_state=RANDOM_STATE, n_jobs=2)
    }
    STACK_CV = 3
    RUN_HYBRID = True

# ---------------------
# Training loop (per target, binary)
# ---------------------
trained = {}
rows = []

for tkey, tcol in TARGETS.items():
    print("\n" + "="*60)
    print("TARGET:", tkey, "| column:", tcol)
    y_series = Y_aug[tcol]
    y_bin, rule = binarize_series(y_series)
    print("Binarize rule:", rule)

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_all, y_bin, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_bin)
    print("Train/test shapes:", X_train.shape, X_test.shape)

    # SMOTE on train
    sm = SMOTE(random_state=RANDOM_STATE)
    X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
    print("After SMOTE class counts:", np.bincount(y_train_res))

    fitted = {}
    # base models
    for name, est in BASE_MODELS.items():
        try:
            m = clone(est)
            t0 = time.time()
            m.fit(X_train_res, y_train_res)
            dur = time.time() - t0
            y_pred = m.predict(X_test)
            y_proba = m.predict_proba(X_test)[:,1] if hasattr(m, "predict_proba") else None
            metrics = evaluate_all(y_test, y_pred, (np.c_[1-y_proba, y_proba] if y_proba is not None else None))
            rows.append({"target": tkey, "model": name, "train_time_s": round(dur,2), **metrics})
            fitted[name] = m
            print(f"  {name}: acc={metrics['accuracy']:.3f} f1={metrics['f1_macro']:.3f} time={dur:.1f}s")
        except Exception as e:
            print(f"  {name} failed:", e)

    # Stack1: fresh estimators
    try:
        xgb_b = XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                              n_estimators=(20 if FAST_MODE else 100),
                              max_depth=(3 if FAST_MODE else 4),
                              random_state=RANDOM_STATE, n_jobs=2)
        rf_b = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=2)
        stack1 = StackingClassifier(
            estimators=[("xgb", xgb_b), ("rf", rf_b)],
            final_estimator=LogisticRegression(max_iter=800),
            cv=STACK_CV, n_jobs=-1
        )
        t0=time.time(); stack1.fit(X_train_res, y_train_res); dur=time.time()-t0
        yp = stack1.predict(X_test); ypb = stack1.predict_proba(X_test) if hasattr(stack1,"predict_proba") else None
        met = evaluate_all(y_test, yp, ypb)
        rows.append({"target": tkey, "model": "Stack1", "train_time_s": round(dur,2), **met})
        fitted["Stack1"] = stack1
        print(f"  Stack1: acc={met['accuracy']:.3f} f1={met['f1_macro']:.3f} time={dur:.1f}s")
    except Exception as e:
        print("  Stack1 failed:", e)

    # Stack2
    try:
        stack2 = StackingClassifier(
            estimators=[("lr", LogisticRegression(max_iter=800, random_state=RANDOM_STATE)),
                        ("gb", GradientBoostingClassifier(random_state=RANDOM_STATE))],
            final_estimator=LogisticRegression(max_iter=800),
            cv=STACK_CV, n_jobs=-1
        )
        t0=time.time(); stack2.fit(X_train_res, y_train_res); dur=time.time()-t0
        yp = stack2.predict(X_test); ypb = stack2.predict_proba(X_test) if hasattr(stack2,"predict_proba") else None
        met = evaluate_all(y_test, yp, ypb)
        rows.append({"target": tkey, "model": "Stack2", "train_time_s": round(dur,2), **met})
        fitted["Stack2"] = stack2
        print(f"  Stack2: acc={met['accuracy']:.3f} f1={met['f1_macro']:.3f} time={dur:.1f}s")
    except Exception as e:
        print("  Stack2 failed:", e)

    # Hybrid (optional)
    if RUN_HYBRID:
        try:
            svm_feat = SVC(kernel="linear", probability=True, random_state=RANDOM_STATE)
            svm_feat.fit(X_train_res, y_train_res)
            try:
                df_train = svm_feat.decision_function(X_train_res)
                df_test  = svm_feat.decision_function(X_test)
            except Exception:
                df_train = svm_feat.predict_proba(X_train_res)[:,1:2]
                df_test  = svm_feat.predict_proba(X_test)[:,1:2]
            if df_train.ndim == 1: df_train = df_train.reshape(-1,1)
            if df_test.ndim == 1: df_test = df_test.reshape(-1,1)
            Xtr_h = np.hstack([X_train_res, df_train])
            Xte_h = np.hstack([X_test, df_test])
            xgb_h = XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                                  n_estimators=(20 if FAST_MODE else 100),
                                  max_depth=(3 if FAST_MODE else 4),
                                  random_state=RANDOM_STATE, n_jobs=2)
            t0=time.time(); xgb_h.fit(Xtr_h, y_train_res); dur=time.time()-t0
            yp = xgb_h.predict(Xte_h); ypb = xgb_h.predict_proba(Xte_h)
            met = evaluate_all(y_test, yp, ypb)
            rows.append({"target": tkey, "model": "Hybrid", "train_time_s": round(dur,2), **met})
            fitted["Hybrid"] = (svm_feat, xgb_h)
            print(f"  Hybrid: acc={met['accuracy']:.3f} f1={met['f1_macro']:.3f} time={dur:.1f}s")
        except Exception as e:
            print("  Hybrid failed:", e)
    else:
        print("  Skipping Hybrid (FAST_MODE)")

    trained[tkey] = {"feature_names": feature_names,
                     "X_train_res": X_train_res, "y_train_res": y_train_res,
                     "X_test": X_test, "y_test": y_test,
                     "models": fitted}

    # free memory a bit
    gc.collect()

# ---------------------
# Save metrics
# ---------------------
perf = pd.DataFrame(rows)
perf.to_csv(os.path.join(ARTIFACTS, "performance_summary.csv"), index=False)
print("\nSaved metrics ->", os.path.join(ARTIFACTS, "performance_summary.csv"))
display(perf)

# ---------------------
# SHAP & LIME (saved to artifacts) - run now in same cell
# ---------------------
print("\nGenerating SHAP and LIME artifacts (XGBoost/LIME) ...")

for tkey, pack in trained.items():
    print("\n--- Explanations for target:", tkey, "---")
    X_test = pack["X_test"]
    X_train_res = pack["X_train_res"]
    feat_names = pack["feature_names"]
    models = pack["models"]

    # SHAP for XGBoost (if available)
    if "XGBoost" in models:
        try:
            xgb = models["XGBoost"]
            explainer = shap.TreeExplainer(xgb)
            shap_vals = explainer.shap_values(X_test)    # binary -> shape (n, f)
            # ensure shapes match
            min_cols = min(shap_vals.shape[1], X_test.shape[1], len(feat_names))
            shap.summary_plot(shap_vals[:, :min_cols], X_test[:, :min_cols], feature_names=feat_names[:min_cols], show=False)
            out_png = os.path.join(ARTIFACTS, f"shap_summary_{tkey}.png")
            plt.tight_layout(); plt.savefig(out_png); plt.close()
            print("  Saved SHAP summary ->", out_png)
            # force plot for first sample
            expected = explainer.expected_value
            fv = shap_vals[0, :min_cols]
            force = shap.force_plot(expected, fv, X_test[0, :min_cols], feature_names=feat_names[:min_cols])
            out_html = os.path.join(ARTIFACTS, f"shap_force_{tkey}.html")
            shap.save_html(out_html, force)
            print("  Saved SHAP force ->", out_html)
        except Exception as e:
            print("  SHAP failed:", e)
    else:
        print("  No XGBoost model for SHAP.")

    # LIME (choose probabilistic model)
    try:
        model_for_lime = None
        if "XGBoost" in models:
            model_for_lime = models["XGBoost"]
        else:
            # pick any with predict_proba
            for m in models.values():
                if hasattr(m, "predict_proba"):
                    model_for_lime = m
                    break
        if model_for_lime is None:
            print("  No probabilistic model available for LIME -> skipping")
        else:
            expl = LimeTabularExplainer(X_train_res, feature_names=feat_names, class_names=["0","1"], discretize_continuous=True)
            explanation = expl.explain_instance(X_test[0], model_for_lime.predict_proba, num_features=10)
            out_txt = os.path.join(ARTIFACTS, f"lime_{tkey}.txt")
            with open(out_txt, "w") as f:
                for feat, val in explanation.as_list():
                    f.write(f"{feat} : {val}\n")
            print("  Saved LIME ->", out_txt)
    except Exception as e:
        print("  LIME failed:", e)

print("\nAll done. Artifacts folder contains plots, HTML and performance CSV.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
=== FULL MODE (full experiments) ===
Raw shape: (196, 39)


,q1,q2,q3,q4,q5,q6,q7,q8,q9,q10,...,q28,q29,q30,q31,q32,q33,q34,q35,q36,Big_five_personality_level
0,1,0,0,0,1,1,0,0,0,0,...,2,3,2,3,1,3,1,1,2,21
1,3,3,2,1,1,1,0,1,1,0,...,2,2,2,3,2,4,2,3,4,26
2,2,2,1,0,0,1,0,1,2,0,...,4,4,1,3,2,4,1,2,1,26
3,0,3,2,0,3,0,1,0,2,0,...,2,3,4,2,3,3,3,3,3,29
4,0,0,0,0,1,1,0,0,0,0,...,0,2,4,2,2,2,3,3,0,22


After dropping ids: (196, 39)
Auto-detected TARGETS: {'depression': 'depression_level', 'life_satisfaction': 'life_satisfication_level', 'personality': 'Big_five_personality_level'}
Original rows: 196
After augmentation rows: 5269
Fitting transformer... (this may take a few seconds)
Transformed shape: (5269, 36) | n_features: 36

TARGET: depression | column: depression_level
Binarize rule: {'rule': 'numeric_median', 'threshold': 14.0}
Train/test shapes: (3688, 36) (1581, 36)
After SMOTE class counts: [1878 1878]
  LogisticRegression: acc=1.000 f1=1.000 time=0.0s
  DecisionTree: acc=0.999 f1=0.999 time=0.2s
  SVM: acc=1.000 f1=1.000 time=0.6s
  GaussianNB: acc=0.982 f1=0.982 time=0.0s
  XGBoost: acc=1.000 f1=1.000 time=1.5s
  Stack1: acc=1.000 f1=1.000 time=10.3s
  Stack2: acc=1.000 f1=1.000 time=24.8s
  Hybrid: acc=1.000 f1=1.000 time=0.1s

TARGET: life_satisfaction | column: life_satisfication_level
Binarize rule: {'rule': 'numeric_median', 'threshold': 16.0}
Train/test shapes: (3688,

,target,model,train_time_s,accuracy,precision_macro,recall_macro,f1_macro,roc_auc_ovr
0,depression,LogisticRegression,0.03,1.000000,1.000000,1.000000,1.000000,1.000000
1,depression,DecisionTree,0.19,0.998735,0.998735,0.998735,0.998735,0.998735
2,depression,SVM,0.63,1.000000,1.000000,1.000000,1.000000,1.000000
3,depression,GaussianNB,0.01,0.981657,0.981724,0.981593,0.981649,0.997975
4,depression,XGBoost,1.52,1.000000,1.000000,1.000000,1.000000,1.000000
5,depression,Stack1,10.28,1.000000,1.000000,1.000000,1.000000,1.000000
6,depression,Stack2,24.81,1.000000,1.000000,1.000000,1.000000,1.000000
7,depression,Hybrid,0.11,1.000000,1.000000,1.000000,1.000000,1.000000
8,life_satisfaction,LogisticRegression,0.02,0.999367,0.999409,0.999321,0.999364,1.000000
9,life_satisfaction,DecisionTree,0.20,0.994940,0.995100,0.994740,0.994914,0.994740



Generating SHAP and LIME artifacts (XGBoost/LIME) ...

--- Explanations for target: depression ---


...

  Saved SHAP summary -> artifacts/shap_summary_depression.png
  Saved SHAP force -> artifacts/shap_force_depression.html
  Saved LIME -> artifacts/lime_depression.txt

--- Explanations for target: life_satisfaction ---


/tmp/ipython-input-210589694.py:360: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_vals[:, :min_cols], X_test[:, :min_cols], feature_names=feat_names[:min_cols], show=False)


  Saved SHAP summary -> artifacts/shap_summary_life_satisfaction.png
  Saved SHAP force -> artifacts/shap_force_life_satisfaction.html
  Saved LIME -> artifacts/lime_life_satisfaction.txt

--- Explanations for target: personality ---


/tmp/ipython-input-210589694.py:360: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_vals[:, :min_cols], X_test[:, :min_cols], feature_names=feat_names[:min_cols], show=False)


  Saved SHAP summary -> artifacts/shap_summary_personality.png
  Saved SHAP force -> artifacts/shap_force_personality.html
  Saved LIME -> artifacts/lime_personality.txt

All done. Artifacts folder contains plots, HTML and performance CSV.


In [ ]:
# =====================================
# Extra: Counterfactuals + XAI Metrics
# =====================================
!pip install -q dice-ml

import dice_ml
from sklearn.metrics import r2_score
from scipy.spatial.distance import cosine


import dice_ml
import random
import seaborn as sns

# -------------------------
# Counterfactual Explanations (DiCE)
# -------------------------
def generate_counterfactuals(model, X_train, y_train, X_test, feat_names, n_cf=3, sample_idx=0):
    """
    Generate counterfactuals using DiCE for one sample.
    """
    df_train = pd.DataFrame(X_train, columns=feat_names)
    df_train["label"] = y_train

    d = dice_ml.Data(dataframe=df_train,
                     continuous_features=feat_names,
                     outcome_name="label")

    m = dice_ml.Model(model=model, backend="sklearn")
    exp = dice_ml.Dice(d, m, method="random")

    x0 = pd.DataFrame([X_test[sample_idx]], columns=feat_names)

    try:
        cf = exp.generate_counterfactuals(x0, total_CFs=n_cf, desired_class="opposite")
        return cf.cf_examples_list[0].final_cfs_df
    except Exception as e:
        print("Counterfactual generation failed:", e)
        return None

# -------------------------
# Interpretability Metrics
# -------------------------
def fidelity_score(model, explainer, X_sample, feat_names, n_features=10):
    try:
        explanation = explainer.explain_instance(X_sample, model.predict_proba, num_features=n_features)
        weights = dict(explanation.as_list())
        approx = sum([X_sample[feat_names.index(f.split(" ")[0])] * w
                      for f, w in weights.items() if f.split(" ")[0] in feat_names])
        true_prob = model.predict_proba([X_sample])[0][1]
        return r2_score([true_prob], [approx])
    except Exception as e:
        print("Fidelity failed:", e)
        return np.nan

def stability_score(model, explainer, X_sample, feat_names, perturb_scale=0.01, n_features=10):
    try:
        exp1 = explainer.explain_instance(X_sample, model.predict_proba, num_features=n_features)
        perturbed = X_sample + np.random.normal(0, perturb_scale, size=len(X_sample))
        exp2 = explainer.explain_instance(perturbed, model.predict_proba, num_features=n_features)

        w1 = dict(exp1.as_list())
        w2 = dict(exp2.as_list())
        common = set(w1.keys()).intersection(set(w2.keys()))
        if not common:
            return np.nan
        v1 = np.array([w1[c] for c in common])
        v2 = np.array([w2[c] for c in common])
        return 1 - cosine(v1, v2)
    except Exception as e:
        print("Stability failed:", e)
        return np.nan

# -------------------------
# Heatmap of Feature Changes
# -------------------------
def plot_agg_heatmap(all_cf, feat_names, title, out_path):
    """
    Plot aggregated heatmap across all models:
    how often each feature changes in any counterfactual.
    """
    # baseline = first row features
    base = all_cf.iloc[0][feat_names]
    changes = (all_cf[feat_names] != base).astype(int)
    freq = changes.sum().sort_values(ascending=False)

    plt.figure(figsize=(10, 6))
    sns.heatmap(freq.to_frame().T, cmap="Blues", annot=True, cbar=False,
                xticklabels=freq.index, yticklabels=["Changed Count"])
    plt.title(title, fontsize=14)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()
    print(f"    Saved aggregated heatmap -> {out_path}")

# -------------------------
# Run on trained targets
# -------------------------
extra_rows = []
N_SAMPLES = 5   # random test samples per model
N_CF = 3        # counterfactuals per sample

for tkey, pack in trained.items():
    print(f"\n>>> Extra XAI metrics for target: {tkey}")
    X_train = pack["X_train_res"]
    y_train = pack["y_train_res"]
    X_test  = pack["X_test"]
    feat_names = pack["feature_names"]
    models = pack["models"]

    # --- Fidelity & Stability (XGBoost only) ---
    if "XGBoost" in models:
        model = models["XGBoost"]
        lime_explainer = LimeTabularExplainer(X_train,
                                              feature_names=feat_names,
                                              class_names=["0","1"],
                                              discretize_continuous=True)

        fid = fidelity_score(model, lime_explainer, X_test[0], feat_names)
        stab = stability_score(model, lime_explainer, X_test[0], feat_names)
        print(f"  Fidelity: {fid:.3f} | Stability: {stab:.3f}")

        extra_rows.append({"target": tkey, "model": "XGBoost",
                           "fidelity": fid, "stability": stab})
    else:
        print("  Skipping: no XGBoost for fidelity/stability")

    # --- Counterfactuals (XGB, SVM, Stack1, Stack2) ---
    selected_models = {k: v for k, v in models.items() if k in ["XGBoost", "SVM", "Stack1", "Stack2"]}
    sample_indices = random.sample(range(len(X_test)), min(N_SAMPLES, len(X_test)))
    all_cf = []

    for mname, model in selected_models.items():
        print(f"  Model: {mname}")
        for idx in sample_indices:
            cf = generate_counterfactuals(model, X_train, y_train, X_test, feat_names,
                                          n_cf=N_CF, sample_idx=idx)
            if cf is not None:
                cf["sample_index"] = idx
                cf["model"] = mname
                cf["target"] = tkey
                all_cf.append(cf)

    if all_cf:
        cf_full = pd.concat(all_cf, ignore_index=True)
        out_csv = os.path.join(ARTIFACTS, f"counterfactuals_{tkey}_ALL.csv")
        cf_full.to_csv(out_csv, index=False)
        print(f"  Saved aggregated counterfactuals -> {out_csv}")

        # --- Aggregated Heatmap ---
        out_png = os.path.join(ARTIFACTS, f"cf_heatmap_{tkey}_ALL.png")
        plot_agg_heatmap(cf_full, feat_names, f"Aggregated Feature Changes in Counterfactuals\nTarget: {tkey}", out_png)

# -------------------------
# Merge with performance_summary.csv
# -------------------------
perf_file = os.path.join(ARTIFACTS, "performance_summary.csv")
if os.path.exists(perf_file):
    perf = pd.read_csv(perf_file)
    perf = perf.merge(pd.DataFrame(extra_rows), on=["target","model"], how="left")
    perf.to_csv(perf_file, index=False)
    print("Updated CSV with interpretability metrics ->", perf_file)
    display(perf)
else:
    print("performance_summary.csv not found!")


>>> Extra XAI metrics for target: depression


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_regression.py:1266: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


  Fidelity: nan | Stability: 0.983
  Model: SVM


  0%|          | 0/1 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but SVC was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but SVC was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but SVC was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but SVC was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but SVC was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but SVC was fitted withou

No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 02 sec
Counterfactual generation failed: No counterfactuals found for any of the query points! Kindly check your configuration.


...

  0%|          | 0/1 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-package

  Model: Stack2


Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but GradientBoostingClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but GradientBoostingClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but GradientBoostingClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/pytho

  Saved aggregated counterfactuals -> artifacts/counterfactuals_personality_ALL.csv
    Saved aggregated heatmap -> artifacts/cf_heatmap_personality_ALL.png
Updated CSV with interpretability metrics -> artifacts/performance_summary.csv


,target,model,train_time_s,accuracy,precision_macro,recall_macro,f1_macro,roc_auc_ovr,fidelity_x,stability_x,fidelity_y,stability_y
0,depression,LogisticRegression,0.03,1.000000,1.000000,1.000000,1.000000,1.000000,NaN,NaN,NaN,NaN
1,depression,DecisionTree,0.19,0.998735,0.998735,0.998735,0.998735,0.998735,NaN,NaN,NaN,NaN
2,depression,SVM,0.63,1.000000,1.000000,1.000000,1.000000,1.000000,NaN,NaN,NaN,NaN
3,depression,GaussianNB,0.01,0.981657,0.981724,0.981593,0.981649,0.997975,NaN,NaN,NaN,NaN
4,depression,XGBoost,1.52,1.000000,1.000000,1.000000,1.000000,1.000000,NaN,0.987637,NaN,0.982792
5,depression,Stack1,10.28,1.000000,1.000000,1.000000,1.000000,1.000000,NaN,NaN,NaN,NaN
6,depression,Stack2,24.81,1.000000,1.000000,1.000000,1.000000,1.000000,NaN,NaN,NaN,NaN
7,depression,Hybrid,0.11,1.000000,1.000000,1.000000,1.000000,1.000000,NaN,NaN,NaN,NaN
8,life_satisfaction,LogisticRegression,0.02,0.999367,0.999409,0.999321,0.999364,1.000000,NaN,NaN,NaN,NaN
9,life_satisfaction,DecisionTree,0.20,0.994940,0.995100,0.994740,0.994914,0.994740,NaN,NaN,NaN,NaN


In [ ]:
# =====================================
# Normalized Percentage Heatmaps (per target)
# =====================================
import seaborn as sns

def plot_percentage_heatmap(cf_csv, feat_names, target, out_path):
    """
    Plot percentage-based heatmap of feature changes across all counterfactuals for a given target.
    """
    cf_full = pd.read_csv(cf_csv)

    # ensure only feature columns are checked
    feat_cols = [c for c in cf_full.columns if c in feat_names]

    # baseline = first row features
    base = cf_full.iloc[0][feat_cols]
    changes = (cf_full[feat_cols] != base).astype(int)

    # percentage of times each feature changed
    freq = (changes.sum() / len(changes)) * 100
    freq = freq.sort_values(ascending=False)

    plt.figure(figsize=(10, 6))
    sns.heatmap(freq.to_frame().T, cmap="Blues", annot=True, fmt=".1f", cbar=False,
                xticklabels=freq.index, yticklabels=["% Changed"])
    plt.title(f"Percentage of Feature Changes in Counterfactuals\nTarget: {target}", fontsize=14)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()
    print(f"Saved percentage heatmap -> {out_path}")

# -------------------------
# Run for all targets
# -------------------------
for tkey, pack in trained.items():
    cf_csv = os.path.join(ARTIFACTS, f"counterfactuals_{tkey}_ALL.csv")
    if os.path.exists(cf_csv):
        out_png = os.path.join(ARTIFACTS, f"cf_heatmap_{tkey}_ALL_percent.png")
        plot_percentage_heatmap(cf_csv, pack["feature_names"], tkey, out_png)
    else:
        print(f"No counterfactuals file found for {tkey}")


Saved percentage heatmap -> artifacts/cf_heatmap_depression_ALL_percent.png
Saved percentage heatmap -> artifacts/cf_heatmap_life_satisfaction_ALL_percent.png
Saved percentage heatmap -> artifacts/cf_heatmap_personality_ALL_percent.png


In [ ]:
# =====================================
# Aggregated Comparison of Top Features Across Targets
# =====================================

def get_feature_change_percentages(cf_csv, feat_names):
    """Return % of feature changes from a counterfactual CSV."""
    cf_full = pd.read_csv(cf_csv)
    feat_cols = [c for c in cf_full.columns if c in feat_names]

    base = cf_full.iloc[0][feat_cols]
    changes = (cf_full[feat_cols] != base).astype(int)
    freq = (changes.sum() / len(changes)) * 100
    return freq.sort_values(ascending=False)

# Collect results
all_results = {}
for tkey, pack in trained.items():
    cf_csv = os.path.join(ARTIFACTS, f"counterfactuals_{tkey}_ALL.csv")
    if os.path.exists(cf_csv):
        freq = get_feature_change_percentages(cf_csv, pack["feature_names"])
        all_results[tkey] = freq
    else:
        print(f"No counterfactuals file for {tkey}")

# ------------------------------------
# Barplots of Top-10 Features Per Target
# ------------------------------------
for tkey, freq in all_results.items():
    top10 = freq.head(10)
    plt.figure(figsize=(8,5))
    sns.barplot(x=top10.values, y=top10.index, palette="Blues_r")
    plt.xlabel("% of Counterfactuals Changed")
    plt.title(f"Top-10 Feature Changes in Counterfactuals\nTarget: {tkey}")
    out_png = os.path.join(ARTIFACTS, f"cf_top10_{tkey}.png")
    plt.tight_layout()
    plt.savefig(out_png)
    plt.close()
    print(f"Saved barplot -> {out_png}")

# ------------------------------------
# Combined Heatmap (all targets)
# ------------------------------------
# Merge into single dataframe
df_compare = pd.DataFrame(all_results).fillna(0)

# Keep only top 10 features overall
top_features = df_compare.mean(axis=1).sort_values(ascending=False).head(10).index
df_top = df_compare.loc[top_features]

plt.figure(figsize=(10,6))
sns.heatmap(df_top, cmap="Blues", annot=True, fmt=".1f", cbar=True,
            xticklabels=df_top.columns, yticklabels=df_top.index)
plt.title("Top Features Across Targets (% Changed in Counterfactuals)", fontsize=14)
plt.xlabel("Target")
plt.ylabel("Feature")
plt.tight_layout()
out_png = os.path.join(ARTIFACTS, "cf_top_features_comparison.png")
plt.savefig(out_png)
plt.close()
print(f"Saved combined heatmap -> {out_png}")


/tmp/ipython-input-1079417494.py:31: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top10.values, y=top10.index, palette="Blues_r")


Saved barplot -> artifacts/cf_top10_depression.png


/tmp/ipython-input-1079417494.py:31: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top10.values, y=top10.index, palette="Blues_r")


Saved barplot -> artifacts/cf_top10_life_satisfaction.png


/tmp/ipython-input-1079417494.py:31: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top10.values, y=top10.index, palette="Blues_r")


Saved barplot -> artifacts/cf_top10_personality.png
Saved combined heatmap -> artifacts/cf_top_features_comparison.png


In [ ]:
# =====================================
# Export Counterfactual Results to LaTeX Tables
# =====================================

# Ensure results are collected
if 'all_results' not in locals():
    all_results = {}
    for tkey, pack in trained.items():
        cf_csv = os.path.join(ARTIFACTS, f"counterfactuals_{tkey}_ALL.csv")
        if os.path.exists(cf_csv):
            freq = get_feature_change_percentages(cf_csv, pack["feature_names"])
            all_results[tkey] = freq

# Export top 10 per target
for tkey, freq in all_results.items():
    top10 = freq.head(10)
    df_tex = pd.DataFrame(top10).reset_index()
    df_tex.columns = ["Feature", "% Changed"]
    out_tex = os.path.join(ARTIFACTS, f"cf_top10_{tkey}.tex")
    with open(out_tex, "w") as f:
        f.write(df_tex.to_latex(index=False, float_format="%.1f"))
    print(f"Saved LaTeX table -> {out_tex}")

# Export combined heatmap table
df_compare = pd.DataFrame(all_results).fillna(0)
top_features = df_compare.mean(axis=1).sort_values(ascending=False).head(10).index
df_top = df_compare.loc[top_features].reset_index()
df_top.rename(columns={"index": "Feature"}, inplace=True)

out_tex = os.path.join(ARTIFACTS, "cf_top_features_comparison.tex")
with open(out_tex, "w") as f:
    f.write(df_top.to_latex(index=False, float_format="%.1f"))
print(f"Saved LaTeX table -> {out_tex}")


Saved LaTeX table -> artifacts/cf_top10_depression.tex
Saved LaTeX table -> artifacts/cf_top10_life_satisfaction.tex
Saved LaTeX table -> artifacts/cf_top10_personality.tex
Saved LaTeX table -> artifacts/cf_top_features_comparison.tex


In [ ]:
# =====================================
# Zip All Artifacts into Single File
# =====================================
import shutil

zip_path = "artifacts_bundle.zip"

# Remove old zip if exists
if os.path.exists(zip_path):
    os.remove(zip_path)

# Zip the entire artifacts folder
shutil.make_archive("artifacts_bundle", "zip", ARTIFACTS)

print(f"All artifacts zipped -> {zip_path}")


All artifacts zipped -> artifacts_bundle.zip
